# 03 Team Analysis

This notebook analyzes constructor-level performance patterns in the Silver layer: points concentration, competitive tiers, season trends, home-country effects, and intra-team balance.

The goal is not to praise one team, but to understand the team-level structure that Gold features should preserve.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "03_team_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 03_team_analysis
Start time: 2026-06-02 00:46:06.824203
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")
meetings = pd.read_parquet(CLEANED_DATA_PATH / "meetings.parquet")

sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)
driver_dim = drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"])
team_results = (
    session_result
    .merge(driver_dim, on=["session_key", "driver_number"], how="left")
    .merge(sessions[["session_key", "year", "event_type", "meeting_key", "country_name", "circuit_short_name"]], on="session_key", how="left", suffixes=("", "_session"))
)
team_results["finish_pos"] = pd.to_numeric(team_results["position"], errors="coerce")
team_results["driver_id"] = team_results["full_name"].fillna("Driver " + team_results["driver_number"].astype(str))
team_results["team_name"] = team_results["team_name"].fillna("Unknown")
team_results["points"] = pd.to_numeric(team_results["points"], errors="coerce").fillna(0)
team_results = team_results.dropna(subset=["session_key", "team_name"])

print(f"Team-driver result rows: {len(team_results):,}")
print(f"Unique teams: {team_results['team_name'].nunique()}")
print(f"Race/Sprint sessions: {team_results['session_key'].nunique()} ({sessions['event_type'].value_counts().to_dict()})")
print(f"Countries: {team_results['country_name'].nunique()}")

Team-driver result rows: 1,374
Unique teams: 13
Race/Sprint sessions: 68 ({'GRAND_PRIX_RACE': 55, 'SPRINT_RACE': 15})
Countries: 21


## 1. Points, Wins, and Podium Concentration

Team points are often more concentrated than driver starts. This section measures how much of the available performance is captured by the leading constructors.

In [3]:
team_session = (
    team_results.groupby(["session_key", "team_name"], as_index=False)
    .agg(
        team_points=("points", "sum"),
        best_finish=("finish_pos", "min"),
        classified_drivers=("finish_pos", lambda s: int(s.notna().sum())),
        drivers_entered=("driver_number", "nunique"),
        year=("year", "first"),
        event_type=("event_type", "first"),
        country_name=("country_name", "first"),
        circuit_short_name=("circuit_short_name", "first"),
    )
)
team_total = (
    team_session.groupby("team_name", as_index=False)
    .agg(
        total_points=("team_points", "sum"),
        sessions_entered=("session_key", "nunique"),
        avg_points_per_session=("team_points", "mean"),
        avg_best_finish=("best_finish", "mean"),
        wins=("best_finish", lambda s: int((s == 1).sum())),
        podium_sessions=("best_finish", lambda s: int((s <= 3).sum())),
    )
)
team_total["podium_session_rate_pct"] = team_total["podium_sessions"] / team_total["sessions_entered"] * 100
team_total["win_session_rate_pct"] = team_total["wins"] / team_total["sessions_entered"] * 100
team_total = team_total.sort_values(["total_points", "wins", "avg_best_finish"], ascending=[False, False, True])
total_points_all = float(team_total["total_points"].sum())
top1_points_pct = float(team_total.head(1)["total_points"].sum() / total_points_all * 100) if total_points_all else 0.0
top3_points_pct = float(team_total.head(3)["total_points"].sum() / total_points_all * 100) if total_points_all else 0.0
team_total.to_csv(OUTPUT_TABLES / "team_points_summary.csv", index=False)
display(team_total)

,team_name,total_points,sessions_entered,avg_points_per_session,avg_best_finish,wins,podium_sessions,podium_session_rate_pct,win_session_rate_pct
7,McLaren,1605.0,68,23.602941,2.415385,26,52,76.470588,38.235294
4,Ferrari,1197.0,68,17.602941,3.687500,6,35,51.470588,8.823529
8,Mercedes,1156.0,68,17.000000,4.205882,13,30,44.117647,19.117647
11,Red Bull Racing,1097.0,68,16.132353,3.602941,23,37,54.411765,33.823529
1,Aston Martin,183.0,68,2.691176,10.584615,0,0,0.000000,0.000000
12,Williams,161.0,68,2.367647,10.753846,0,3,4.411765,0.000000
5,Haas F1 Team,156.0,68,2.294118,10.044776,0,0,0.000000,0.000000
0,Alpine,122.0,68,1.794118,11.676471,0,1,1.470588,0.000000
10,Racing Bulls,113.0,38,2.973684,9.270270,0,1,2.631579,0.000000
6,Kick Sauber,74.0,60,1.233333,12.600000,0,1,1.666667,0.000000


In [4]:
fig = px.bar(
    team_total.sort_values("total_points"),
    x="total_points",
    y="team_name",
    orientation="h",
    color="podium_session_rate_pct",
    title="Team Points and Podium Rate",
    labels={"team_name": "Team", "total_points": "Total points", "podium_session_rate_pct": "Podium session rate %"},
)
fig.write_html(OUTPUT_CHARTS / "team_points_summary.html", include_plotlyjs="cdn")
fig.show()

In [5]:
concentration = team_total.sort_values("total_points", ascending=True).copy()
concentration["cumulative_points_pct"] = concentration["total_points"].cumsum() / total_points_all * 100 if total_points_all else 0
concentration["teams_pct"] = (np.arange(len(concentration)) + 1) / len(concentration) * 100
concentration.to_csv(OUTPUT_TABLES / "team_points_concentration.csv", index=False)
fig = px.line(concentration, x="teams_pct", y="cumulative_points_pct", title="Team Points Concentration")
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", name="Equal distribution")
fig.write_html(OUTPUT_CHARTS / "team_points_concentration.html", include_plotlyjs="cdn")
fig.show()

## 2. Competitive Hierarchy

Rather than hard-coding team labels, this section assigns tiers from the observed point distribution. The exact labels are descriptive and should be treated as EDA categories, not permanent business rules.

In [6]:
ranked = team_total.copy()
ranked["points_rank_pct"] = ranked["total_points"].rank(pct=True, method="first")
ranked["tier"] = pd.cut(
    ranked["points_rank_pct"],
    bins=[0, 1/3, 2/3, 1.0],
    labels=["Backmarker", "Midfield", "Top"],
    include_lowest=True,
)
tier_stats = ranked.groupby("tier", observed=False).agg(
    teams=("team_name", "count"),
    avg_points=("total_points", "mean"),
    avg_podium_rate=("podium_session_rate_pct", "mean"),
    avg_wins=("wins", "mean"),
).reset_index()
ranked.to_csv(OUTPUT_TABLES / "team_tiers.csv", index=False)
tier_stats.to_csv(OUTPUT_TABLES / "team_tier_stats.csv", index=False)
display(ranked[["team_name", "total_points", "wins", "podium_session_rate_pct", "tier"]])
display(tier_stats)

,team_name,total_points,wins,podium_session_rate_pct,tier
7,McLaren,1605.0,26,76.470588,Top
4,Ferrari,1197.0,6,51.470588,Top
8,Mercedes,1156.0,13,44.117647,Top
11,Red Bull Racing,1097.0,23,54.411765,Top
1,Aston Martin,183.0,0,0.000000,Top
12,Williams,161.0,0,4.411765,Midfield
5,Haas F1 Team,156.0,0,0.000000,Midfield
0,Alpine,122.0,0,1.470588,Midfield
10,Racing Bulls,113.0,0,2.631579,Midfield
6,Kick Sauber,74.0,0,1.666667,Backmarker


,tier,teams,avg_points,avg_podium_rate,avg_wins
0,Backmarker,4,30.5,0.416667,0.0
1,Midfield,4,138.0,2.128483,0.0
2,Top,5,1047.6,45.294118,13.6


In [7]:
fig = px.bar(
    ranked.sort_values("total_points"),
    x="total_points",
    y="team_name",
    color="tier",
    orientation="h",
    title="Observed Team Competitive Tiers",
)
fig.write_html(OUTPUT_CHARTS / "team_tiers.html", include_plotlyjs="cdn")
fig.show()

## 3. Team Performance Over Seasons

A team feature should capture both level and trajectory. This section tracks season points, average finish, and year-over-year point changes.

In [8]:
team_season = (
    team_results.groupby(["team_name", "year"], as_index=False)
    .agg(
        season_points=("points", "sum"),
        avg_finish=("finish_pos", "mean"),
        sessions=("session_key", "nunique"),
        wins=("finish_pos", lambda s: int((s == 1).sum())),
        podiums=("finish_pos", lambda s: int((s <= 3).sum())),
    )
)
team_season = team_season[team_season["sessions"] >= 3].sort_values(["team_name", "year"])
team_season["points_change_pct"] = team_season.groupby("team_name")["season_points"].pct_change() * 100
team_season["finish_improvement"] = team_season.groupby("team_name")["avg_finish"].shift(1) - team_season["avg_finish"]
team_season.to_csv(OUTPUT_TABLES / "team_season_trends.csv", index=False)
display(team_season.head(30))

,team_name,year,season_points,avg_finish,sessions,wins,podiums,points_change_pct,finish_improvement
0,Alpine,2024,65.0,12.388889,30,0,2,NaN,NaN
1,Alpine,2025,22.0,14.673077,30,0,0,-66.153846,-2.284188
2,Alpine,2026,35.0,10.400000,8,0,0,59.090909,4.273077
3,Aston Martin,2024,94.0,11.981818,30,0,0,NaN,NaN
4,Aston Martin,2025,89.0,11.680000,30,0,0,-5.319149,0.301818
5,Aston Martin,2026,0.0,16.444444,8,0,0,-100.000000,-4.764444
6,Audi,2026,2.0,12.100000,8,0,0,NaN,NaN
7,Cadillac,2026,0.0,16.538462,8,0,0,NaN,NaN
8,Ferrari,2024,652.0,4.333333,30,5,25,NaN,NaN
9,Ferrari,2025,398.0,6.264151,30,1,9,-38.957055,-1.930818


In [9]:
top_teams = team_total.head(8)["team_name"].tolist()
trend_plot = team_season[team_season["team_name"].isin(top_teams)]
fig = px.line(
    trend_plot,
    x="year",
    y="season_points",
    color="team_name",
    markers=True,
    title="Season Points Trend for Leading Teams",
)
fig.write_html(OUTPUT_CHARTS / "team_season_points_trend.html", include_plotlyjs="cdn")
fig.show()

In [10]:
volatility = (
    team_season.groupby("team_name", as_index=False)
    .agg(points_std=("season_points", "std"), points_mean=("season_points", "mean"), seasons=("year", "nunique"))
)
volatility["points_cv_pct"] = volatility["points_std"] / volatility["points_mean"] * 100
volatility = volatility.sort_values("points_cv_pct")
volatility.to_csv(OUTPUT_TABLES / "team_points_volatility.csv", index=False)
display(volatility)

,team_name,points_std,points_mean,seasons,points_cv_pct
8,Mercedes,144.049760,385.333333,3,37.383156
0,Alpine,22.052967,40.666667,3,54.228606
5,Haas F1 Team,30.446675,52.000000,3,58.551297
4,Ferrari,252.501485,399.000000,3,63.283580
7,McLaren,380.792594,535.000000,3,71.176186
11,Red Bull Racing,276.074869,365.666667,3,75.499053
1,Aston Martin,52.886671,61.000000,3,86.699461
10,Racing Bulls,50.204581,56.500000,2,88.857666
6,Kick Sauber,46.669048,37.000000,2,126.132561
12,Williams,72.341781,53.666667,3,134.798350


## 4. Home-Country Signal

Home advantage is a weak and sparse signal in this dataset because many constructors are multinational and not every home country appears every season. The analysis is recorded as a review signal, not a hard feature rule.

In [11]:
home_countries = {
    "Red Bull Racing": ["Austria"],
    "Ferrari": ["Italy"],
    "Mercedes": ["Great Britain", "Germany"],
    "McLaren": ["Great Britain"],
    "Aston Martin": ["Great Britain"],
    "Williams": ["Great Britain"],
    "Alpine": ["France"],
    "Haas F1 Team": ["United States"],
    "RB": ["Italy"],
    "Kick Sauber": ["Switzerland"],
    "Sauber": ["Switzerland"],
}
team_results["home_country_match"] = team_results.apply(
    lambda row: row["country_name"] in home_countries.get(row["team_name"], []),
    axis=1,
)
home_away = (
    team_results.dropna(subset=["finish_pos"])
    .groupby(["team_name", "home_country_match"], as_index=False)
    .agg(avg_finish=("finish_pos", "mean"), avg_points=("points", "mean"), rows=("driver_number", "count"))
)
home_pivot = home_away.pivot(index="team_name", columns="home_country_match", values=["avg_finish", "avg_points", "rows"])
home_pivot.columns = [f"{metric}_{'home' if flag else 'away'}" for metric, flag in home_pivot.columns]
home_pivot = home_pivot.reset_index()
if {"avg_finish_home", "avg_finish_away"}.issubset(home_pivot.columns):
    home_pivot["home_finish_delta"] = home_pivot["avg_finish_away"] - home_pivot["avg_finish_home"]
else:
    home_pivot["home_finish_delta"] = np.nan
home_pivot.to_csv(OUTPUT_TABLES / "home_country_signal.csv", index=False)
display(home_pivot.sort_values("home_finish_delta", ascending=False, na_position="last"))

,team_name,avg_finish_away,avg_finish_home,avg_points_away,avg_points_home,rows_away,rows_home,home_finish_delta
4,Ferrari,5.211864,4.125000,9.279661,12.750000,118.0,8.0,1.086864
9,RB,12.470588,12.000000,0.882353,0.333333,51.0,3.0,0.470588
5,Haas F1 Team,11.669903,11.454545,1.339806,0.818182,103.0,22.0,0.215357
11,Red Bull Racing,6.608333,7.400000,8.933333,5.000000,120.0,5.0,-0.791667
0,Alpine,13.123967,NaN,1.008264,NaN,121.0,NaN,NaN
1,Aston Martin,12.201754,NaN,1.605263,NaN,114.0,NaN,NaN
2,Audi,12.100000,NaN,0.200000,NaN,10.0,NaN,NaN
3,Cadillac,16.538462,NaN,0.000000,NaN,13.0,NaN,NaN
6,Kick Sauber,14.186916,NaN,0.691589,NaN,107.0,NaN,NaN
7,McLaren,3.864000,NaN,12.840000,NaN,125.0,NaN,NaN


In [12]:
plot_home = home_pivot.dropna(subset=["home_finish_delta"]).copy()
fig = px.bar(
    plot_home.sort_values("home_finish_delta"),
    x="home_finish_delta",
    y="team_name",
    orientation="h",
    title="Home-Country Finish Delta (positive means better at home)",
)
fig.write_html(OUTPUT_CHARTS / "home_country_signal.html", include_plotlyjs="cdn")
fig.show()

## 5. Intra-Team Balance

A balanced team has two drivers contributing similar points. This section measures how much the lower-scoring driver contributes relative to the higher-scoring driver within each team.

In [13]:
driver_team_points = (
    team_results.groupby(["team_name", "driver_id"], as_index=False)
    .agg(driver_points=("points", "sum"), sessions=("session_key", "nunique"), avg_finish=("finish_pos", "mean"))
)
balance_rows = []
for team_name, group in driver_team_points.groupby("team_name"):
    active = group.sort_values("driver_points", ascending=False)
    if len(active) < 2:
        continue
    top_two = active.head(2)
    high = float(top_two.iloc[0]["driver_points"])
    low = float(top_two.iloc[1]["driver_points"])
    balance_rows.append({
        "team_name": team_name,
        "lead_driver": top_two.iloc[0]["driver_id"],
        "second_driver": top_two.iloc[1]["driver_id"],
        "lead_points": high,
        "second_points": low,
        "points_balance_pct": low / high * 100 if high else np.nan,
        "total_top_two_points": high + low,
    })
team_balance = pd.DataFrame(balance_rows).sort_values("points_balance_pct", ascending=False)
team_balance.to_csv(OUTPUT_TABLES / "team_driver_balance.csv", index=False)
display(team_balance)

,team_name,lead_driver,second_driver,lead_points,second_points,points_balance_pct,total_top_two_points
10,Racing Bulls,Liam LAWSON,Isack HADJAR,54.0,51.0,94.444444,105.0
7,McLaren,Lando NORRIS,Oscar PIASTRI,855.0,750.0,87.719298,1605.0
12,Williams,Alexander ALBON,Carlos SAINZ,86.0,70.0,81.395349,156.0
5,Haas F1 Team,Oliver BEARMAN,Nico HULKENBERG,60.0,41.0,68.333333,101.0
1,Aston Martin,Fernando ALONSO,Lance STROLL,126.0,57.0,45.238095,183.0
8,Mercedes,George RUSSELL,Kimi ANTONELLI,652.0,281.0,43.098160,933.0
4,Ferrari,Charles LECLERC,Carlos SAINZ,673.0,290.0,43.090639,963.0
9,RB,Yuki TSUNODA,Daniel RICCIARDO,30.0,12.0,40.000000,42.0
6,Kick Sauber,Nico HULKENBERG,Gabriel BORTOLETO,51.0,19.0,37.254902,70.0
0,Alpine,Pierre GASLY,Esteban OCON,84.0,23.0,27.380952,107.0


In [14]:
fig = px.bar(
    team_balance.sort_values("points_balance_pct"),
    x="points_balance_pct",
    y="team_name",
    orientation="h",
    color="total_top_two_points",
    title="Intra-Team Points Balance",
    labels={"points_balance_pct": "Second driver points as % of lead driver", "team_name": "Team"},
)
fig.write_html(OUTPUT_CHARTS / "team_driver_balance.html", include_plotlyjs="cdn")
fig.show()

## Final Team Analysis Report

The final report summarizes team structure signals for later Gold feature engineering.

In [15]:
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "teams_analyzed": int(team_total["team_name"].nunique()),
    "sessions_analyzed": int(team_session["session_key"].nunique()),
    "total_points": total_points_all,
    "top1_points_concentration_pct": top1_points_pct,
    "top3_points_concentration_pct": top3_points_pct,
    "top_team": team_total.iloc[0]["team_name"] if len(team_total) else None,
    "top_team_points": float(team_total.iloc[0]["total_points"]) if len(team_total) else 0.0,
    "team_season_rows": int(len(team_season)),
    "home_signal_teams": int(home_pivot["home_finish_delta"].notna().sum()) if "home_finish_delta" in home_pivot else 0,
    "avg_driver_balance_pct": float(team_balance["points_balance_pct"].mean()) if not team_balance.empty else 0.0,
}
write_report("team_analysis", report)
write_insight(
    "Silver Team Analysis Insights",
    [
        f"Analyzed {report['teams_analyzed']} teams across {report['sessions_analyzed']} race/sprint sessions.",
        f"Top team: {report['top_team']} with {report['top_team_points']:.0f} points.",
        f"Top three teams account for {report['top3_points_concentration_pct']:.1f}% of points.",
        f"Average top-two driver balance is {report['avg_driver_balance_pct']:.1f}%.",
    ],
    [],
    [
        "Use team points tier, season trend, and intra-team balance as candidate Gold features.",
        "Treat home-country advantage as sparse review signal rather than a mandatory model feature.",
        "Keep team identity joined by session_key and driver_number to avoid driver/team mapping drift.",
    ],
)
(CHECKPOINTS / "silver_team_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '03_team_analysis', 'timestamp': '2026-06-02T00:46:08.817900', 'teams_analyzed': 13, 'sessions_analyzed': 68, 'total_points': 5912.0, 'top1_points_concentration_pct': 27.148173207036535, 'top3_points_concentration_pct': 66.94857916102842, 'top_team': 'McLaren', 'top_team_points': 1605.0, 'team_season_rows': 31, 'home_signal_teams': 4, 'avg_driver_balance_pct': 48.735443096993976}
